In [30]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sksurv.metrics import brier_score
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator
from lifelines import KaplanMeierFitter
from sksurv.util import Surv
import wandb
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

### Data preprocessing

In [31]:
# Load data
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')
meta = pd.read_csv('Data/metaData.csv')

# Converting to dataframe
train = pd.DataFrame(train)
test = pd.DataFrame(test)

# Splitting train and test into X_train and y_train
X_train = train.drop(columns=['event', 'event_id', 'time_to_hit_hours'])
y_train = train[['time_to_hit_hours', 'event']]
y_train['event'] = y_train['event'].astype(int)

# y_train for gbsa
y_train_surv = Surv.from_arrays(event = y_train["event"].astype(bool), 
                                time = y_train["time_to_hit_hours"])
event_vals = y_train['event'].copy() # Later use for oof prediction loop
time_vals = y_train['time_to_hit_hours'].copy()

# Dropping event_id for X_test
X_test = test.drop(columns=['event_id'])
event_id = test['event_id'] # Save event_id for submitting

# Horizon
HORIZONS = [12, 24, 48, 72]
DROP_HORIZONS = [24, 48, 72]


In [32]:
# Helper function
def compute_brier_sc(time, event, preds, horizons):
    brier_sc = np.zeros(len(horizons))

    for i, horizon in enumerate(horizons):
        valid = ~((event == 0) & (time < horizon))
        y_true = ((event == 1) & (time <= horizon)).astype(int)
        brier_sc[i] = np.mean((y_true[valid] - preds[valid, i]) ** 2)

    return brier_sc

def compute_hybrid_sc(time, event, preds, horizons):
    risk_score = preds[:, 1] * 0.3 + preds[:, 2] * 0.4 + preds[:, 3] * 0.3
    c_index = concordance_index_censored(
        event.astype(bool),
        time,
        risk_score
    )[0]
    brier_score = compute_brier_sc(time, event, preds, horizons)
    p24 = brier_score[1]
    p48 = brier_score[2]
    p72 = brier_score[3]
    weighted_brier = 0.3*p24 + 0.4*p48 + 0.3*p72
    hybrid_score = 0.3*c_index + 0.7*(1 - weighted_brier)
    return hybrid_score

### Gradient Boost Survival Analysis

In [33]:
# Total folds
N_FOLDS_GBSA = 5

# Initialize oof matrix prediction
oof_preds_gbsa = np.zeros((len(X_train), 4)) # 4 horizons: 12h, 24h, 48h, 72h

# Initialize final average test prediction
gbsa_test_preds = np.zeros((len(X_test), 4))

# Initialize 10 random seeds
GBSA_SEEDS = [42, 67, 69, 2026, 2006, 458, 478, 456, 782, 126, 48, 29, 10, 500, 600]

# More config
gbsa_configs = [
    {"learning_rate":0.010, "subsample":0.70, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.010, "subsample":0.85, "max_depth":3, "min_samples_leaf":15,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.010, "subsample":0.60, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.005, "subsample":0.85, "max_depth":3, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":2000},
    {"learning_rate":0.010, "subsample":0.85, "max_depth":3, "min_samples_leaf":20,  "min_samples_split":3, "n_estimators":1400},
    {"learning_rate":0.008, "subsample":0.75, "max_depth":2, "min_samples_leaf":15,  "min_samples_split":4, "n_estimators":1500},
    {"learning_rate":0.015, "subsample":0.70, "max_depth":3, "min_samples_leaf":10,  "min_samples_split":3, "n_estimators":1000},
    {"learning_rate":0.005, "subsample":0.90, "max_depth":3, "min_samples_leaf":18,  "min_samples_split":5, "n_estimators":2500},
    {"learning_rate":0.010, "subsample":0.80, "max_depth":4, "min_samples_leaf":12,  "min_samples_split":3, "n_estimators":1200},
    {"learning_rate":0.020, "subsample":0.65, "max_depth":3, "min_samples_leaf":10,  "min_samples_split":3, "n_estimators":800},
]

gbsa_configs = gbsa_configs[: 2]
GBSA_SEEDS = GBSA_SEEDS[:2]

##### Train GBSA

In [34]:
for config in gbsa_configs:
    for seed in GBSA_SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS_GBSA, shuffle=True, random_state=seed)
        seed_test_preds = np.zeros((len(X_test), len(HORIZONS)))

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, event_vals)):
            X_tr  = X_train.iloc[train_idx].copy()
            X_val = X_train.iloc[val_idx].copy()
            y_tr_surv  = y_train_surv[train_idx]

            # Initializing gbsa model
            gbsa = GradientBoostingSurvivalAnalysis(**config, random_state=seed)
            gbsa.fit(X_tr, y_tr_surv)

            # OOF
            val_preds_fold = np.zeros((len(val_idx), len(HORIZONS)))
            for i, sf in enumerate(gbsa.predict_survival_function(X_val)):
                for j, horizon in enumerate(HORIZONS):
                    val_preds_fold[i, j] = 1 - sf(min(horizon, sf.x[-1])) # type: ignore
            oof_preds_gbsa[val_idx] += val_preds_fold

            # Test predictions
            for i, sf in enumerate(gbsa.predict_survival_function(X_test)):
                for j, h in enumerate(HORIZONS):
                    seed_test_preds[i, j] += (1 - sf(min(h, sf.x[-1]))) # type: ignore

        # Average across folds for this (config, seed) pair
        gbsa_test_preds += seed_test_preds / N_FOLDS_GBSA

# Average across all (config, seed) pairs
total_runs = len(gbsa_configs) * len(GBSA_SEEDS)
oof_preds_gbsa  /= total_runs
gbsa_test_preds /= total_runs

oof_preds_gbsa = np.clip(oof_preds_gbsa, 0, 1)
gbsa_test_preds = np.clip(gbsa_test_preds, 0, 1)

In [35]:
risk_score = oof_preds_gbsa[:, 1] * 0.3 + oof_preds_gbsa[:, 2] * 0.4 + oof_preds_gbsa[:, 3] * 0.3
risk_score.shape

(221,)

### GBSA evaluation (c_index, brier score)

In [36]:
c_index = concordance_index_censored(
    event_vals.astype(bool),
    time_vals,
    risk_score
)[0]
brier_score = compute_brier_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)
hybrid_sc = compute_hybrid_sc(time_vals, event_vals, oof_preds_gbsa, HORIZONS)

print(f"C_index: {c_index:.4f}\n"
      f"Brier p24h: {brier_score[1]:.4f}\n"
      f"Brier p48h: {brier_score[2]:.4f}\n"
      f"Brier p72h: {brier_score[3]:.4f}\n" # Dell care
      f"Hybrid score: {hybrid_sc:.4f}\n")

C_index: 0.9446
Brier p24h: 0.0272
Brier p48h: 0.0146
Brier p72h: 0.0010
Hybrid score: 0.9734



### Train lgbm (12 24 48)

In [37]:
# Initialize N_folds for lgbm
N_FOLDS_LGBM = 5

# Initialize random seeds for kFOLD
LGBM_SEEDS = [526, 484, 749, 852, 848, 123, 456, 789, 147, 369, 165, 986, 8945, 8946, 592]

# Horizon
HORIZONS = [12, 24, 48, 72]
DROP_HORIZONS_72 = [12, 24, 48]

# Initialize oof matrix prediction
oof_preds_lgbm = np.zeros((len(X_train), 3))

# Initialize final average test prediction
lgbm_test_preds = np.zeros((len(X_test), 3))

# For safety
X_train_lgbm = X_train.values
X_test_lgbm = X_test.values

# Adding configs
lgb_cfgs = {
    12: {  # 12h: very few events -> heavy regularization
        "max_depth":2, "learning_rate":0.03, "n_estimators":200,
        "subsample":0.7, "colsample_bytree":0.7, "min_child_samples":10,
        "reg_alpha":1.0, "reg_lambda":3.0, "num_leaves":4,
    },
    24: {  # 24h: moderate events -> balanced
        "max_depth":3, "learning_rate":0.03, "n_estimators":300,
        "subsample":0.7, "colsample_bytree":0.7, "min_child_samples":8,
        "reg_alpha":0.5, "reg_lambda":2.0, "num_leaves":7,
    },
    48: {  # 48h: more events -> slightly less regularization
        "max_depth":2, "learning_rate":0.05, "n_estimators":200,
        "subsample":0.8, "colsample_bytree":0.8, "min_child_samples":5,
        "reg_alpha":0.1, "reg_lambda":1.0, "num_leaves":4,
    },
}

###### ai rảnh thì refactor cái này giùm t tks you<3

In [39]:
for i, horizon in enumerate(DROP_HORIZONS_72):    

    y_train_lgbm = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    uncensored_row = ~((event_vals == 0) & (time_vals < horizon))
    uncensored_idx = np.where(uncensored_row)[0]
    censored_idx = np.where(~uncensored_row)[0] # Fill in later with predict_proba
    horizon_oof = np.zeros(len(X_train_lgbm))
    test_oof = np.zeros(len(X_test_lgbm))

    for seed in LGBM_SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS_LGBM, shuffle=True, random_state=seed)
        seed_oof_preds = np.zeros(len(X_train_lgbm))
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(uncensored_idx, y_train_lgbm[uncensored_row])):
            valid_train_idx = uncensored_idx[train_idx] # Hết biết đặt tên gì r
            valid_val_idx = uncensored_idx[val_idx]

            X_tr = X_train_lgbm[valid_train_idx].copy()
            X_val = X_train_lgbm[valid_val_idx].copy()
            y_tr = y_train_lgbm.iloc[valid_train_idx].copy()

            # Calculate the IPCW
            train_times = y_train['time_to_hit_hours'].iloc[valid_train_idx].values
            train_events = y_train['event'].iloc[valid_train_idx].values

            kmf_fold = KaplanMeierFitter()
            kmf_fold.fit(
                durations=train_times,
                event_observed=(1 - train_events)  # type: ignore
            )
            ipcw_weights = np.ones(len(valid_train_idx))
            for k in range(len(train_idx)):
                if train_events[k] == 1:
                    g_t = kmf_fold.survival_function_at_times(train_times[k]).values[0]
                    ipcw_weights[k] = 1.0 / max(g_t, 1e-6)


            lgbm = LGBMClassifier(**lgb_cfgs[horizon], random_state=seed, verbose=-1)
            lgbm.fit(
                X_tr, y_tr,
                sample_weight=ipcw_weights
            )
            seed_oof_preds[valid_val_idx] = lgbm.predict_proba(X_val)[:, 1] # type: ignore

        seed_oof_preds[censored_idx] = last_model.predict_proba(X_train_lgbm[censored_idx])[:, 1] # type: ignore
        horizon_oof += seed_oof_preds    

        # Full train on X_test
        full_times  = y_train['time_to_hit_hours'].iloc[uncensored_idx].values
        full_events = y_train['event'].iloc[uncensored_idx].values
        kmf_full = KaplanMeierFitter()
        kmf_full.fit(durations=full_times, event_observed=(1 - full_events)) # type: ignore
        ipcw_full = np.ones(len(uncensored_idx))
        for k in range(len(uncensored_idx)):
            if full_events[k] == 1:
                g_t = kmf_full.survival_function_at_times(full_times[k]).values[0]
                ipcw_full[k] = 1.0 / max(g_t, 1e-6)
        lgbm_full = LGBMClassifier(**lgb_cfgs[horizon], random_state=seed, verbose=-1)
        lgbm_full.fit(X_train_lgbm[uncensored_idx], y_train_lgbm.iloc[uncensored_idx],
                 sample_weight=ipcw_full
        )
        test_oof += lgbm_full.predict_proba(X_test_lgbm)[:, 1] # type: ignore

    oof_preds_lgbm[:, i] = horizon_oof / len(LGBM_SEEDS)
    lgbm_test_preds[:, i] = test_oof / len(LGBM_SEEDS)

### LGBM Evaluation

In [42]:
brier_score = compute_brier_sc(time_vals, event_vals, oof_preds_lgbm, DROP_HORIZONS_72)
print(f"Brier p12h: {brier_score[0]:.4f}\n" # Dell care
      f"Brier p24h: {brier_score[1]:.4f}\n"
      f"Brier p48h: {brier_score[2]:.4f}\n") 

Brier p12h: 0.0566
Brier p24h: 0.0337
Brier p48h: 0.0222



### Blend weight

In [ ]:
final_preds = []

### Final submissions 

In [44]:
gbsa_test_preds[:, 3] = 1.0
submission = {
    'event_id' : event_id,
    'prob_12h' : lgbm_test_preds[:, 0],# type: ignore
    'prob_24h' : lgbm_test_preds[:, 1],# type: ignore
    'prob_48h' : lgbm_test_preds[:, 2],# type: ignore
    'prob_72h' : gbsa_test_preds[:, 3] # type: ignore
}
submission = pd.DataFrame(submission)
submission.to_csv('final_preds.csv', index=False)